# Raw `Decisão` values per court group

Loads every file from `data/1_raw/`, extracts the `Decisão` line from the `Observacoes` field.

In [ ]:
import json
import re
from pathlib import Path

DATA_DIR = Path('../../data/1_raw')

COURTS = ['STJ', 'STA', 'TCAN', 'TCAS', 'TRC', 'TRE', 'TRG', 'TRL', 'TRP']

def load_court(name):
    path = DATA_DIR / f'{name}.json'
    with open(path, encoding='utf-8') as f:
        return json.load(f)

def extract_decisao(observacoes):
    """Extract the Decisão value from an Observacoes string."""
    if not observacoes:
        return None
    match = re.search(r'Decis[ãa]o:\s*(.+)', observacoes)
    if match:
        return match.group(1).split('\n')[0].strip()
    return None

data = {}
for court in COURTS:
    records = load_court(court)
    data[court] = records
    print(f'{court}: {len(records)} records')

## Group 1 — STJ / STA

In [ ]:
GROUP1 = ['STJ', 'STA']
count = 0
for court in GROUP1:
    print(f'\n{'='*60}')
    print(f'  {court}')
    print(f'{'='*60}')
    for i, record in enumerate(data[court]):
        decisao = extract_decisao(record.get('Observacoes'))
        if decisao != None:
            count += 1
        print(f'  [{i}] {decisao}')

## Group 2 — TCAN / TCAS

In [ ]:
GROUP2 = ['TCAN', 'TCAS']
count = 0
for court in GROUP2:
    print(f'\n{'='*60}')
    print(f'  {court}')
    print(f'{'='*60}')
    for i, record in enumerate(data[court]):
        decisao = extract_decisao(record.get('Observacoes'))
        if decisao != None:
            count += 1
        print(f'  [{i}] {decisao}')

## Group 3 — TRC / TRE / TRG / TRL / TRP

In [ ]:
GROUP3 = ['TRC', 'TRE', 'TRG', 'TRL', 'TRP']
count = 0
for court in GROUP3:
    print(f'\n{'='*60}')
    print(f'  {court}')
    print(f'{'='*60}')
    for i, record in enumerate(data[court]):
        decisao = extract_decisao(record.get('Observacoes'))
        if decisao != None:
            count += 1
        print(f'  [{i}] {decisao}')

## Top 20 `Decisão` — by group

In [ ]:
from collections import Counter

GROUPS = {
    'STJ / STA':             ['STJ', 'STA'],
    'TCAN / TCAS':           ['TCAN', 'TCAS'],
    'TRC / TRE / TRG / TRL / TRP': ['TRC', 'TRE', 'TRG', 'TRL', 'TRP'],
}

for group_name, courts in GROUPS.items():
    all_decisoes = []
    for court in courts:
        for record in data[court]:
            d = extract_decisao(record.get('Observacoes'))
            if d is not None:
                all_decisoes.append(d)

    counter = Counter(all_decisoes)
    top20 = counter.most_common(20)

    print(f'\n{"="*60}')
    print(f'  {group_name}  (total with Decisão: {len(all_decisoes)})')
    print(f'{"="*60}')
    for rank, (value, count) in enumerate(top20, 1):
        print(f'  {rank:2}. ({count:6,})  {value}')

In [ ]:
import pandas as pd

rows = []
for court in COURTS:
    total = len(data[court])
    with_decisao = sum(
        1 for r in data[court]
        if extract_decisao(r.get('Observacoes')) is not None
    )
    rows.append({
        'Tribunal': court,
        'Total registos': total,
        'Com Decisão': with_decisao,
        'Sem Decisão': total - with_decisao,
        '% Com Decisão': round(with_decisao / total * 100, 1),
    })

df_counts = pd.DataFrame(rows).set_index('Tribunal')
display(df_counts)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

GROUP3 = ['TRC', 'TRE', 'TRG', 'TRL', 'TRP']

g3 = df_counts.loc[GROUP3]
has_dec  = g3['Com Decisão'].values
no_dec   = g3['Sem Decisão'].values
pcts     = g3['% Com Decisão'].values
x        = np.arange(len(GROUP3))

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(x, has_dec, label='With Decisão')
ax.bar(x, no_dec, bottom=has_dec, color='#A9A9A9', alpha=0.55, label='Without Decisão')

totals = has_dec + no_dec
for xi, pct, total in zip(x, pcts, totals):
    ax.text(xi, total + totals.max() * 0.015,
            f'{pct}%', ha='center', va='bottom',
            fontsize=9.5, fontweight='bold', color='#222')

ax.set_xticks(x)
ax.set_xticklabels(GROUP3, fontsize=11)
ax.set_ylabel('Records', fontsize=10)
ax.set_title('Decisão Coverage per Court — Group 3',
             fontsize=13, fontweight='bold', pad=12)
ax.set_ylim(0, totals.max() * 1.14)
ax.legend(fontsize=9, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)
ax.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.6)
ax.set_axisbelow(True)

fig.tight_layout()
plt.savefig('decisao_coverage_grupo3.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada em notebooks/decisao_coverage_grupo3.png')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

GROUPS = {
    'STJ / STA':                   ['STJ', 'STA'],
    'TCAN / TCAS':                 ['TCAN', 'TCAS'],
    'TRC / TRE / TRG / TRL / TRP': ['TRC', 'TRE', 'TRG', 'TRL', 'TRP'],
}

group_frames = {}
for group_name, courts in GROUPS.items():
    vals = [
        extract_decisao(r.get('Observacoes'))
        for court in courts
        for r in data[court]
    ]
    vals = [v for v in vals if v is not None]
    top20 = pd.Series(vals).value_counts().head(20).sort_values()
    group_frames[group_name] = top20

COLORS = ['#4C72B0', '#55A868', '#C44E52']
fig, axes = plt.subplots(1, 3, figsize=(20, 8))
fig.suptitle('Top 20 Decisão por grupo de tribunais', fontsize=15, fontweight='bold', y=1.01)

for ax, (group_name, series), color in zip(axes, group_frames.items(), COLORS):
    bars = ax.barh(series.index, series.values, color=color, alpha=0.85, edgecolor='white')

    for bar, val in zip(bars, series.values):
        ax.text(
            bar.get_width() + series.values.max() * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f'{val:,}',
            va='center', ha='left', fontsize=8.5, color='#333333'
        )

    ax.set_title(group_name, fontsize=11, fontweight='bold', pad=8)
    ax.set_xlabel('Nº de ocorrências', fontsize=9)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.tick_params(axis='y', labelsize=8.5)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_xlim(right=series.values.max() * 1.18)

plt.tight_layout()
plt.savefig('top20_decisao.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada em top20_decisao.png')

In [ ]:
import numpy as np

GROUPS = {
    'STJ / STA':                   ['STJ', 'STA'],
    'TCAN / TCAS':                 ['TCAN', 'TCAS'],
    'TRC / TRE / TRG / TRL / TRP': ['TRC', 'TRE', 'TRG', 'TRL', 'TRP'],
}

PALETTES = {
    'STJ / STA':                   ['#2166ac', '#74add1'],
    'TCAN / TCAS':                 ['#1a9850', '#a6d96a'],
    'TRC / TRE / TRG / TRL / TRP': ['#d73027', '#f46d43', '#fdae61', '#fee090', '#c7a0d8'],
}

fig, axes = plt.subplots(1, 3, figsize=(22, 9))
fig.suptitle('Top 20 Decisão por grupo — distribuição por tribunal', fontsize=14, fontweight='bold', y=1.01)

for ax, (group_name, courts) in zip(axes, GROUPS.items()):
    court_series = {}
    all_vals = []
    for court in courts:
        vals = [
            extract_decisao(r.get('Observacoes'))
            for r in data[court]
        ]
        vals = [v for v in vals if v is not None]
        court_series[court] = pd.Series(vals).value_counts()
        all_vals.extend(vals)

    top20_labels = pd.Series(all_vals).value_counts().head(20).sort_values().index.tolist()

    df = pd.DataFrame(
        {court: court_series[court] for court in courts},
        index=top20_labels
    ).fillna(0)

    palette = PALETTES[group_name]
    lefts = np.zeros(len(top20_labels))

    for court, color in zip(courts, palette):
        vals = df[court].values
        ax.barh(top20_labels, vals, left=lefts, color=color, alpha=0.88,
                edgecolor='white', linewidth=0.5, label=court)
        lefts += vals

    totals = df.sum(axis=1).values
    for i, total in enumerate(totals):
        ax.text(total + totals.max() * 0.01, i, f'{int(total):,}',
                va='center', ha='left', fontsize=8, color='#333333')

    ax.set_title(group_name, fontsize=10.5, fontweight='bold', pad=8)
    ax.set_xlabel('Nº de ocorrências', fontsize=9)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.tick_params(axis='y', labelsize=8.5)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_xlim(right=totals.max() * 1.18)
    ax.legend(fontsize=8.5, loc='lower right', framealpha=0.7)

plt.tight_layout()
plt.savefig('top20_decisao_por_tribunal.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada em top20_decisao_por_tribunal.png')

In [ ]:
GROUPS_ORDERED = [
    ('STJ / STA',                   ['STJ', 'STA']),
    ('TCAN / TCAS',                 ['TCAN', 'TCAS']),
    ('TRC / TRE / TRG / TRL / TRP', ['TRC', 'TRE', 'TRG', 'TRL', 'TRP']),
]

all_courts = [c for _, courts in GROUPS_ORDERED for c in courts]
GROUP_COLORS = {
    'STJ': '#2166ac', 'STA': '#74add1',
    'TCAN': '#1a9850', 'TCAS': '#a6d96a',
    'TRC': '#d73027', 'TRE': '#f46d43', 'TRG': '#fdae61', 'TRL': '#a5c4e0', 'TRP': '#c7a0d8',
}
GROUP_LABELS = {c: g for g, courts in GROUPS_ORDERED for c in courts}

TOP_N = 20
fig, axes = plt.subplots(1, 9, figsize=(38, 9))
fig.suptitle('Top 20 Decisão — por tribunal individual', fontsize=14, fontweight='bold', y=1.01)

for ax, court in zip(axes, all_courts):
    vals = [
        extract_decisao(r.get('Observacoes'))
        for r in data[court]
    ]
    vals = [v for v in vals if v is not None]

    if vals:
        series = pd.Series(vals).value_counts().head(TOP_N).sort_values()
        bars = ax.barh(series.index, series.values,
                       color=GROUP_COLORS[court], alpha=0.88, edgecolor='white')
        for bar, val in zip(bars, series.values):
            ax.text(bar.get_width() + series.values.max() * 0.02,
                    bar.get_y() + bar.get_height() / 2,
                    f'{int(val):,}', va='center', ha='left', fontsize=7.5, color='#333333')
        ax.set_xlim(right=series.values.max() * 1.25)
    else:
        ax.text(0.5, 0.5, 'Sem dados', transform=ax.transAxes,
                ha='center', va='center', fontsize=9, color='grey')

    group_label = GROUP_LABELS[court]
    ax.set_title(f'{court}\n({group_label})', fontsize=8.5, fontweight='bold', pad=6)
    ax.set_xlabel('Nº ocorrências', fontsize=7.5)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.tick_params(axis='y', labelsize=7.5)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('top20_decisao_por_tribunal_individual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada em top20_decisao_por_tribunal_individual.png')